# 1 IMPORT LIBRARY

In [74]:
import pandas as pd
import numpy as np
import os
import sys
import subprocess
import time
from datetime import datetime, timedelta
import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas

In [75]:
folder_path = "O:\\Shared drives\\VEMP Supply Planning\\Projects\\MRP Transformation\\MRP_BOM_Central_official\\"

In [ ]:
con = snowflake.connector.connect(

)
cur = con.cursor()

Initiating login request with your identity provider. A browser window should have opened for you to complete the login. If you can't see it, check existing browser windows, or your OS settings. Press CTRL+C to abort and try again...
Going to open: https://sso.colpal.com/app/snowflake/exk5d2wdao1LeZ8Vr357/sso/saml?SAMLRequest=pZNLb%2BIwFIX%2FSuRZJ04CpcUCqvBqI4VCS4pmujOJAQvHTn0dQv99HR6jzqLdzM6yz%2FF37z127%2F5YCOfANHAl%2ByjwfOQwmamcy20fvaZT9w45YKjMqVCS9dEHA3Q%2F6AEtREmiyuzkC3uvGBjHXiSBNAd9VGlJFAUORNKCATEZWUazhISeTygA08bi0MWSA7esnTElwbiua69ueUpvcej7Pva72KoayS%2F0BVH%2BzCi1MipT4mo52p6%2BQQTYbzcIq7CExcU45PI8gp8o67MIyGOaLtzFfJkiJ7p2N1ISqoLpJdMHnrHXl%2BRcANgKRvPkIUoniyiZzZN4NRlPVl4Fbsak0VQE3jYrPZCq3gi6Z5kqyspYjmdXeMNyLNSW2%2BnF4z4q9zxvP08el6PqmT51dlU6G%2B43sVk%2BmD9r2k1%2Fd8P37HA7pWJVDdfbDDmra9Zhk3UMULFYNgkbu%2BWHLTfw3bCT%2BgFpdUi76wXt7htyxjZhLqk5Oa9tAChbkyipOJVGyxL%2FrRqz4%2F4mD%2BucqiBhb3cr3bq5bRy4iQ%2BdXxA54fXg%2F%2BfSw1%2FvuzzPJ5tYPF4owbMPZ6p0Qc33gQZecNrhubs5SQkrKBdRnmsGYIMVQtUjzaixv8Doii

# 2 DOWNLOAD ZMNU

In [77]:
# subprocess.check_call(['C:\Program Files (x86)\SAP\FrontEnd\SAPgui\\sapshcut.exe', '-system=CAP', '-client=321', '-user=USERNAME', '-pw=PASSWORD'])
# time.sleep(20)
# print ('Done')

In [78]:
# vbs_path = folder_path + "zmnu_bom_all_level.vbs"
# subprocess.call(['cscript.exe', vbs_path])
# print ('Done ZMNU')

# 3 TRANSFORM ZMNU INTO BOMTREE

## 3.1 READ ZMNU

In [79]:
df= pd.read_csv(folder_path+"BOM_All_Level_do_not_delete.xls",sep= "\t",encoding="utf-16",skiprows=3,usecols=list(range(1,20)))

df = df[~df['Material'].isnull()]
df = df[~df['Material'].str.startswith('CMK')]

column_mapping = {
    df.columns[4]: 'Component Quantity',
    df.columns[8]: 'Material Quantity',
    df.columns[5]: 'Component UOM',
    df.columns[10]: 'Material UOM'
}

df = df.rename(columns=column_mapping)
df.columns = df.columns.str.strip()

# Get recycle material
df_recycle = df[df['Material'].str.match(r'^M\d{5}$')].copy()

# Filter BOM status 1
df = df[(df["BOM Sts"] == 1) & (~df["Material"].str.match(r'^M\d{5}$'))]

# Filter out CmpValidFr > year 2100
df = df[df['CmpValidFr'].str[-4:].astype(int) < 2100]

# Get material code data
df_pair = df.iloc[:, [0, 2]].copy()
df_pair = df_pair.drop_duplicates()

In [80]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 11606 entries, 0 to 29661
Data columns (total 19 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Material               11606 non-null  object 
 1   Material Description   11606 non-null  object 
 2   Component              11606 non-null  object 
 3   Component Description  11606 non-null  object 
 4   Component Quantity     11606 non-null  object 
 5   Component UOM          11606 non-null  object 
 6   AltBOM                 11606 non-null  object 
 7   BOM Sts                11606 non-null  int64  
 8   Material Quantity      11606 non-null  object 
 9   Item                   11606 non-null  object 
 10  Material UOM           11606 non-null  object 
 11  Change no.             4367 non-null   float64
 12  chng no. to            1164 non-null   float64
 13  OpScrap%               11606 non-null  float64
 14  cmpScrap%              11606 non-null  float64
 15  BO

## 3.2 EXTRACT ECN CHANGE

In [81]:
df_ecn = df[['Material','Component','Change no.','chng no. to','CmpValidFr','cmpValidTo']].copy()
df_ecn['Change no.'] = df_ecn['Change no.'].astype(str)
df_ecn['Change no.'] = df_ecn['Change no.'].str.rstrip('.0')
df_ecn['chng no. to'] = df_ecn['chng no. to'].astype(str)
df_ecn['chng no. to'] = df_ecn['chng no. to'].str.rstrip('.0')
df_ecn.head()

,Material,Component,Change no.,chng no. to,CmpValidFr,cmpValidTo
0,31371,C0000000934,nan,nan,05/25/2016,12/31/9999
1,31371,P11230075,nan,nan,05/25/2016,12/31/9999
2,31372,C0000001016,nan,nan,05/25/2016,12/31/9999
3,31372,P11230075,nan,nan,05/25/2016,12/31/9999
4,31373,C0000001017,nan,nan,05/25/2016,12/31/9999


In [82]:
# Calculate today - 1 day
one_day_ago = datetime.now() - timedelta(days=1)
one_day_ago_str = one_day_ago.strftime('%m/%d/%Y')
one_day_ago_dt = datetime.strptime(one_day_ago_str, '%m/%d/%Y')

In [83]:
# Filter the DataFrame based on string comparison
df_ecn_filtered_temp = df_ecn[
    df_ecn['cmpValidTo'].apply(lambda x: datetime.strptime(x, '%m/%d/%Y')) >= one_day_ago_dt
]
df_ecn_filtered_temp = df_ecn_filtered_temp[df_ecn_filtered_temp['cmpValidTo'].str[-4] < '3000']
#filtered_df = filtered_df[~filtered_df['cmpValidTo'].str[-4].eq('9999')]
df_ecn_filtered_temp = df_ecn_filtered_temp[df_ecn_filtered_temp['chng no. to']!='nan']

ECN_no = df_ecn_filtered_temp['chng no. to'].unique()
ECN_no

array(['500000028368', '500000028679', '500000028369', '50000002868',
       '500000028681', '500000027084', '500000028682', '500000028683',
       '500000028684', '500000028685', '500000028686', '500000028687',
       '500000028688', '500000028689', '500000027092', '50000002706',
       '500000027061', '500000027062', '500000027058', '500000027059',
       '50000002869', '500000028691', '500000027065', '500000027064',
       '500000027093', '500000028364', '500000028365', '500000029074',
       '500000028366', '500000028367', '50000002837', '500000028371',
       '500000028372', '500000028374', '500000028693', '500000028694',
       '500000028695', '500000028696', '500000028375', '500000028697',
       '500000028314', '500000027063', '500000028698', '500000028699',
       '5000000287', '500000027069', '500000027095', '500000028702',
       '500000028703', '500000028704', '500000028705', '500000028706',
       '50000002707', '500000027072', '500000027073', '500000028378',
       '50000

In [84]:
# Create a boolean mask for rows where 'Change no.' or 'chng no. to' is in ECN_no
mask = (df_ecn['Change no.'].isin(ECN_no)) | (df_ecn['chng no. to'].isin(ECN_no))

# Apply the mask to the DataFrame to filter rows
df_ecn_filtered = df_ecn[mask]
df_ecn_filtered

,Material,Component,Change no.,chng no. to,CmpValidFr,cmpValidTo
326,312272,C0000005899,nan,500000028368,03/10/2023,04/17/2024
327,312272,C0000010544,500000028368,nan,04/17/2024,12/31/9999
328,312272,C0000005900,nan,500000028368,03/10/2023,04/17/2024
329,312272,C0000010545,500000028368,nan,04/17/2024,12/31/9999
330,312272,C0000005901,nan,500000028368,03/10/2023,04/17/2024
...,...,...,...,...,...,...
29323,VN01129A,C0000010371,500000029326,nan,11/26/2023,12/31/9999
29327,VN01129A,P11270110,nan,500000028821,02/14/2019,10/27/2023
29328,VN01129A,P11270195,500000028821,nan,10/27/2023,12/31/9999
29460,VN01267A,P11270110,500000016876,500000028822,10/23/2018,10/27/2023


In [85]:
# Replace "nan" strings with actual NaN values
df_ecn_filtered.replace('nan', np.nan, inplace=True)

# Create dictionaries to store Component values for matching Change no. and chng no. to
change_to_component = {}
chng_to_change = {}

# Create a dictionary to store the "New Material" values in the original order
new_material_dict = {}

# Function to create the "New/old Material" column
def create_new_material(row):
    new_material = []

    if not pd.isna(row['Change no.']):
        change_no = row['Change no.']
        
        matching_rows = df_ecn_filtered[df_ecn_filtered['chng no. to'] == change_no]
        if not matching_rows.empty:
            components = matching_rows['Component'].tolist()
            if len(components) > 1:
                new_material.extend(components)
            elif len(components) == 1:
                new_material.append(components[0])

    if not pd.isna(row['chng no. to']):
        chng_no_to = row['chng no. to']
        
        matching_rows = df_ecn_filtered[df_ecn_filtered['Change no.'] == chng_no_to]
        if not matching_rows.empty:
            components = matching_rows['Component'].tolist()
            if len(components) > 1:
                new_material.extend(components)
            elif len(components) == 1:
                new_material.append(components[0])

    if new_material:
        new_material_dict[row.name] = ' - '.join(new_material)
        return ' - '.join(new_material)
    else:
        return row['Component']

# Apply the function to create the new column
df_ecn_filtered['New/old Material'] = df_ecn_filtered.apply(create_new_material, axis=1)

# Reorder the "New Material" column based on the original order
df_ecn_filtered.loc[:, 'New/old Material'] = df_ecn_filtered.index.map(new_material_dict)

df_ecn_filtered

C:\Users\Danh Bui\AppData\Local\Temp\ipykernel_11480\1100895899.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_ecn_filtered.replace('nan', np.nan, inplace=True)
C:\Users\Danh Bui\AppData\Local\Temp\ipykernel_11480\1100895899.py:44: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_ecn_filtered['New/old Material'] = df_ecn_filtered.apply(create_new_material, axis=1)
C:\Users\Danh Bui\AppData\Local\Temp\ipykernel_11480\1100895899.py:47: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_

,Material,Component,Change no.,chng no. to,CmpValidFr,cmpValidTo,New/old Material
326,312272,C0000005899,NaN,500000028368,03/10/2023,04/17/2024,C0000010544 - C0000010545 - C0000010546 - C000...
327,312272,C0000010544,500000028368,NaN,04/17/2024,12/31/9999,C0000005899 - C0000005900 - C0000005901 - C000...
328,312272,C0000005900,NaN,500000028368,03/10/2023,04/17/2024,C0000010544 - C0000010545 - C0000010546 - C000...
329,312272,C0000010545,500000028368,NaN,04/17/2024,12/31/9999,C0000005899 - C0000005900 - C0000005901 - C000...
330,312272,C0000005901,NaN,500000028368,03/10/2023,04/17/2024,C0000010544 - C0000010545 - C0000010546 - C000...
...,...,...,...,...,...,...,...
29323,VN01129A,C0000010371,500000029326,NaN,11/26/2023,12/31/9999,C0000005930 - C0000005931 - C0000005932 - C000...
29327,VN01129A,P11270110,NaN,500000028821,02/14/2019,10/27/2023,P11270195
29328,VN01129A,P11270195,500000028821,NaN,10/27/2023,12/31/9999,P11270110
29460,VN01267A,P11270110,500000016876,500000028822,10/23/2018,10/27/2023,P11270195


## 3.3 CREATE BOMTREE

In [86]:
# Convert data to a list of tuples
list_pair = list(df_pair.itertuples(index=False, name=None))
list_pair

[('31371', 'C0000000934'),
 ('31371', 'P11230075'),
 ('31372', 'C0000001016'),
 ('31372', 'P11230075'),
 ('31373', 'C0000001017'),
 ('31373', 'P11230075'),
 ('31375', 'C0000001018'),
 ('31375', 'P11230075'),
 ('31377', 'C0000002130'),
 ('31377', 'P11230082'),
 ('31380', 'C0000002485'),
 ('31380', 'P11230086'),
 ('31381', 'C0000002486'),
 ('31381', 'P11230086'),
 ('31382', 'C0000002487'),
 ('31382', 'P11230086'),
 ('31387', 'C0000003540'),
 ('31387', 'P11230086'),
 ('31388', 'C0000003790'),
 ('31388', 'P11230082'),
 ('31390', 'C0000003791'),
 ('31390', 'P11230082'),
 ('31392', 'C0000003792'),
 ('31392', 'P11230082'),
 ('31394', 'C0000003793'),
 ('31394', 'P11230082'),
 ('31452', 'C0000005691'),
 ('31452', 'P11230082'),
 ('31454', 'C0000005692'),
 ('31454', 'P11230082'),
 ('31455', 'C0000005693'),
 ('31455', 'P11230082'),
 ('31463', 'C0000003537'),
 ('31463', 'P11230082'),
 ('31464', 'C0000003538'),
 ('31464', 'P11230082'),
 ('31465', 'C0000003539'),
 ('31465', 'P11230082'),
 ('31511', '

In [87]:
class TreeNode:
    def __init__(self, value):
        self.value = value
        self.children = []

def build_tree(list_pair, parent, tree):
    children = [child for parent_name, child in list_pair if parent_name == parent.value]

    for child_value in children:
        child_node = TreeNode(child_value)
        parent.children.append(child_node)
        build_tree(list_pair, child_node, tree)

def get_tree_hierarchy(node, path=[]):
    path.append(node.value)
    hierarchy = []

    if not node.children:
        hierarchy.append(path.copy())

    for child in node.children:
        hierarchy.extend(get_tree_hierarchy(child, path.copy()))

    path.pop()
    return hierarchy


unique_parents = set(parent for parent, _ in list_pair)
unique_children = set(child for _, child in list_pair)

root_names = list(unique_parents - unique_children)

root_nodes = [TreeNode(name) for name in root_names]

for root in root_nodes:
    build_tree(list_pair, root, root)

hierarchies = []
for root in root_nodes:
    hierarchies.extend(get_tree_hierarchy(root))

# Convert the hierarchies list into a DataFrame
BOMtree = pd.DataFrame(hierarchies)

In [88]:
BOMtree

,0,1,2,3,4,5
0,1051620,C0000002485,C0000002941,M43203,None,None
1,1051620,C0000002485,C0000002941,M22711,None,None
2,1051620,C0000002485,M37819,None,None,None
3,1051620,C0000002485,M31079,None,None,None
4,1051620,C0000002485,M31436,None,None,None
...,...,...,...,...,...,...
28726,C0000002089,M30710,None,None,None,None
28727,C0000002089,C0000002086,M21165,None,None,None
28728,C0000002089,C0000002086,M36803,None,None,None
28729,C0000002089,C0000002086,C0000002083,M21165,None,None


## 3.4 MERGE BOMTREE WITH MARC

In [89]:
sql_marc = 'SELECT * FROM "SBX_SC_HUB"."MP3_DATA"."MP3_MARC"'
cur.execute(sql_marc)
df_marc = cur.fetch_pandas_all()
df_marc

,MATERIAL,PLANT,MAINTENANCE_STATUS,BATCH_MANAGEMENT,PLANT_SP_MATL_STATUS,VALID_FROM,MRP_TYPE,MRP_CONTROLLER,PLANNED_DELIV_TIME,GR_PROCESSING_TIME,...,COVERAGE_PROFILE,SELECTION_METHOD,INDIVIDUAL_COLL,PROD_SCHED_PROFILE,VERSION_INDICATOR,UNDERDELY_TOLERANCE,OVERDELY_TOLERANCE,UNLTD_OVERDELIVERY,ABC_INDICATOR,INSPECTION_PLAN_IND
0,C0000000987,VN11,LGBDESA,,C5,None,ND,TBD,0,0,...,,2,2,VN11IM,X,0.0,0.0,X,,
1,C0000000990,VN11,LBGDESV,,C5,None,ND,TBD,0,0,...,,2,2,,X,0.0,0.0,,,
2,C0000000994,VN11,LBGDESV,,C5,None,ND,TBD,0,0,...,,2,2,,X,0.0,0.0,,,
3,P11270056,VN11,DBGLESQV,X,C3,None,PD,TBI,20,1,...,,2,2,,,0.0,0.0,,A,
4,Z105BE00281,VN11,EDL,,12,None,ND,SP1,30,0,...,,,,,,0.0,0.0,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12104,P11212357,VN11,LSBGQED,X,C8,None,PD,TB4,7,1,...,,2,2,,,0.0,0.0,,B,
12105,P11212446,VN11,LSQBGED,X,C8,None,PD,TB4,8,1,...,,2,2,,,0.0,0.0,,B,
12106,P11212059,VN11,LQSBGED,X,C5,None,ND,TBD,7,1,...,,2,2,,,0.0,0.0,,C,
12107,P11212060,VN11,LQSBGED,X,C5,None,ND,TBD,7,1,...,,2,2,,,0.0,0.0,,C,


In [90]:
BOMtree = BOMtree.merge(df_marc[['MATERIAL', 'PLANT_SP_MATL_STATUS','PLANT']], how='left', left_on=0, right_on='MATERIAL')
BOMtree = BOMtree[~(BOMtree["PLANT_SP_MATL_STATUS"] == 'C5')]
BOMtree = BOMtree.drop(columns=['MATERIAL', 'PLANT_SP_MATL_STATUS','PLANT'])
BOMtree

,0,1,2,3,4,5
65,VN01289A,C0000009305,M22411,None,None,None
66,VN01289A,C0000009305,M44034,None,None,None
67,VN01289A,C0000009305,M44761,None,None,None
68,VN01289A,C0000009305,M44030,None,None,None
69,VN01289A,C0000009305,C0000009257,M27839,None,None
...,...,...,...,...,...,...
28668,FVN00156,P11241721,None,None,None,None
28669,FVN00156,P11270073,None,None,None,None
28670,FVN00156,P11270120,None,None,None,None
28671,FVN00156,P11270110,None,None,None,None


## 3.5 EXTRACT CONSUMPTION RATIO

In [91]:
df_ratio = df.iloc[:, [0,8,2,4]].copy()
df_ratio

,Material,Material Quantity,Component,Component Quantity
0,31371,"1,000.000",C0000000934,"1,000.000"
1,31371,"1,000.000",P11230075,"1,000.000"
2,31372,"1,000.000",C0000001016,"1,000.000"
3,31372,"1,000.000",P11230075,"1,000.000"
4,31373,"1,000.000",C0000001017,"1,000.000"
...,...,...,...,...
29657,VN01327A,"1,000.000",P11211726,"144,000.000"
29658,VN01327A,"1,000.000",P11220265,548.370
29659,VN01327A,"1,000.000",P11241570,"1,000.000"
29660,VN01327A,"1,000.000",P11270106,1.200


In [92]:
df_ratio['Component Quantity'] = df_ratio['Component Quantity'].str.replace(',', '').astype(float)
df_ratio['Material Quantity'] = df_ratio['Material Quantity'].str.replace(',', '').astype(float)
df_ratio['Ratio'] = df_ratio['Component Quantity'] / df_ratio['Material Quantity']
df_ratio = df_ratio[['Material','Component','Ratio']]

### LIST SUM

In [93]:
df_list_sum = pd.read_csv(folder_path + "List material cum sum.csv", delimiter=',')
df_list_sum = df_list_sum.drop_duplicates()
df_list_sum['Sum']='Yes'
df_list_sum

,Material,Component,Sum
0,C0000002130,M30701,Yes
1,C0000005544,M21165,Yes
3,C0000005545,M21165,Yes
5,C0000005546,M21165,Yes
7,C0000009992,M38273,Yes
8,C0000009992,M19934,Yes
11,C0000009993,M38273,Yes
12,C0000009993,M19934,Yes
15,C0000009994,M38273,Yes
16,C0000009994,M19934,Yes


In [94]:
df_ratio = df_ratio.merge(df_list_sum, how='left', on=['Material','Component'])
df_ratio['Ratio'] = np.where(df_ratio['Sum'] == 'Yes', df_ratio.groupby(['Material', 'Component','Sum'])['Ratio'].transform('sum'), df_ratio['Ratio'])
df_ratio = df_ratio.drop('Sum', axis=1)
df_ratio

,Material,Component,Ratio
0,31371,C0000000934,1.00000
1,31371,P11230075,1.00000
2,31372,C0000001016,1.00000
3,31372,P11230075,1.00000
4,31373,C0000001017,1.00000
...,...,...,...
11601,VN01327A,P11211726,144.00000
11602,VN01327A,P11220265,0.54837
11603,VN01327A,P11241570,1.00000
11604,VN01327A,P11270106,0.00120


In [95]:
# Find the index of the rows with the highest ratio within each group
idx = df_ratio.groupby(['Material', 'Component'])['Ratio'].idxmax()

# Select rows using the index
df_ratio_max = df_ratio.loc[idx]

# Reset the index if needed
df_ratio_max = df_ratio_max.reset_index(drop=True)

df_ratio_max

,Material,Component,Ratio
0,1050301,C0000004023,18.00000
1,1050301,C0000004024,18.00000
2,1050301,C0000004025,18.00000
3,1050301,C0000004026,18.00000
4,1050301,P11211060,72.00000
...,...,...,...
11476,VN01327A,P11211726,144.00000
11477,VN01327A,P11220265,0.54837
11478,VN01327A,P11241570,1.00000
11479,VN01327A,P11270073,0.00116


In [96]:
BOMtree.columns = ['lv1', 'lv2', 'lv3', 'lv4', 'lv5', 'lv6']
BOMtree = BOMtree.fillna('null')
BOMtree

,lv1,lv2,lv3,lv4,lv5,lv6
65,VN01289A,C0000009305,M22411,null,null,null
66,VN01289A,C0000009305,M44034,null,null,null
67,VN01289A,C0000009305,M44761,null,null,null
68,VN01289A,C0000009305,M44030,null,null,null
69,VN01289A,C0000009305,C0000009257,M27839,null,null
...,...,...,...,...,...,...
28668,FVN00156,P11241721,null,null,null,null
28669,FVN00156,P11270073,null,null,null,null
28670,FVN00156,P11270120,null,null,null,null
28671,FVN00156,P11270110,null,null,null,null


## 3.6 CREATE BOMTREE WITH RATIO

In [97]:
BOMtree_ratio = BOMtree.copy()

In [98]:
def left_merge_dataframes_ratio(left_col1, left_col2):
    global BOMtree_ratio
    
    result_df = pd.merge(BOMtree_ratio, df_ratio_max, how='left', left_on=[left_col1, left_col2], right_on=['Material', 'Component'])
    
    # Drop 'Material' and 'Component' columns
    result_df = result_df.drop(columns=['Material', 'Component'])
    
    # Rename columns with suffixes based on the input column names
    result_df = result_df.rename(columns={
        'Ratio': 'Ratio_' + left_col2
    })
    
    BOMtree_ratio = result_df  # Update the global variable
    return BOMtree_ratio

# Call the function for each pair of columns
left_merge_dataframes_ratio('lv1', 'lv2')
left_merge_dataframes_ratio('lv2', 'lv3')
left_merge_dataframes_ratio('lv3', 'lv4')
left_merge_dataframes_ratio('lv4', 'lv5')
left_merge_dataframes_ratio('lv5', 'lv6')

,lv1,lv2,lv3,lv4,lv5,lv6,Ratio_lv2,Ratio_lv3,Ratio_lv4,Ratio_lv5,Ratio_lv6
0,VN01289A,C0000009305,M22411,null,null,null,18.000000,0.000076,NaN,NaN,NaN
1,VN01289A,C0000009305,M44034,null,null,null,18.000000,0.000510,NaN,NaN,NaN
2,VN01289A,C0000009305,M44761,null,null,null,18.000000,0.000151,NaN,NaN,NaN
3,VN01289A,C0000009305,M44030,null,null,null,18.000000,0.000250,NaN,NaN,NaN
4,VN01289A,C0000009305,C0000009257,M27839,null,null,18.000000,1.000000,0.002646,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
13505,FVN00156,P11241721,null,null,null,null,1.000000,NaN,NaN,NaN,NaN
13506,FVN00156,P11270073,null,null,null,null,0.000900,NaN,NaN,NaN,NaN
13507,FVN00156,P11270120,null,null,null,null,0.012500,NaN,NaN,NaN,NaN
13508,FVN00156,P11270110,null,null,null,null,0.006188,NaN,NaN,NaN,NaN


## 3.7 RELAYOUT

### DEAL WITH 3*

In [99]:
# Create an empty dictionary to store the result
BOMtree_temp = {}

# Iterate through DataFrame rows
for index, row in BOMtree_ratio.iterrows():
    BOMtree_temp[index] = []

    # Iterate through columns for each row
    for col in range(len(row)):
        value = row[col]

        # Check if the value is not None and not NaN (if NaN values exist)
        if pd.notna(value) and value is not None:
            BOMtree_temp[index].append(value)

# Create an empty dictionary for the transformed data
BOMtree_transformed = {}

# Iterate through the BOMtree data and apply the transformation rule
for key, values in BOMtree_temp.items():
    if values[1] is not None and values[1].startswith('3'):
        BOMtree_transformed[key] = values  # Keep the same as origin
    else:
        # Rearrange the column positions as specified
        transformed_values = [values[0]] + [None] * 1  # Initialize with 5 elements
        transformed_values.extend(values[1:])  # Add remaining elements from the original data
        BOMtree_transformed[key] = transformed_values

# Print the transformed data
for key, values in BOMtree_transformed.items():
    print(key, values)

0 ['VN01289A', None, 'C0000009305', 'M22411', 'null', 'null', 'null', 18.0, 7.6e-05]
1 ['VN01289A', None, 'C0000009305', 'M44034', 'null', 'null', 'null', 18.0, 0.00051]
2 ['VN01289A', None, 'C0000009305', 'M44761', 'null', 'null', 'null', 18.0, 0.00015099999999999998]
3 ['VN01289A', None, 'C0000009305', 'M44030', 'null', 'null', 'null', 18.0, 0.00025]
4 ['VN01289A', None, 'C0000009305', 'C0000009257', 'M27839', 'null', 'null', 18.0, 1.0, 0.002646]
5 ['VN01289A', None, 'C0000009305', 'C0000009257', 'M37998', 'null', 'null', 18.0, 1.0, 5.4e-05]
6 ['VN01289A', None, 'C0000009305', 'C0000009257', 'C0000009240', 'M29142', 'null', 18.0, 1.0, 1.0, 0.000279]
7 ['VN01289A', None, 'C0000009305', 'C0000009257', 'C0000009240', 'M38272', 'null', 18.0, 1.0, 1.0, 0.00451]
8 ['VN01289A', None, 'C0000009305', 'C0000009257', 'C0000009240', 'M40443', 'null', 18.0, 1.0, 1.0, 0.00451]
9 ['VN01289A', None, 'C0000009306', 'M22411', 'null', 'null', 'null', 18.0, 7.6e-05]
10 ['VN01289A', None, 'C0000009306', 

In [100]:
# Convert the dictionary to a list of lists with padding
max_length = max(len(row) for row in BOMtree_transformed.values())
data_list = [row + [None] * (max_length - len(row)) for row in BOMtree_transformed.values()]

# Create a DataFrame from the list of lists
BOMtree_transformed = pd.DataFrame(data_list)

# Reset column names (optional)
BOMtree_transformed.columns = ['SKU', '3****', 'C toothbrush', 'C HDL', 'C HDL S.1', 'Raw Material','ratio2','ratio3','ratio4','ratio5','ratio6']

BOMtree_transformed.replace({None: 'null', np.nan: 'null'}, inplace=True)

In [101]:
BOMtree_transformed

,SKU,3****,C toothbrush,C HDL,C HDL S.1,Raw Material,ratio2,ratio3,ratio4,ratio5,ratio6
0,VN01289A,null,C0000009305,M22411,null,null,null,18.000000,0.000076,null,null
1,VN01289A,null,C0000009305,M44034,null,null,null,18.000000,0.00051,null,null
2,VN01289A,null,C0000009305,M44761,null,null,null,18.000000,0.000151,null,null
3,VN01289A,null,C0000009305,M44030,null,null,null,18.000000,0.00025,null,null
4,VN01289A,null,C0000009305,C0000009257,M27839,null,null,18.000000,1.0,0.002646,null
...,...,...,...,...,...,...,...,...,...,...,...
13505,FVN00156,null,P11241721,null,null,null,null,1.000000,null,null,null
13506,FVN00156,null,P11270073,null,null,null,null,0.000900,null,null,null
13507,FVN00156,null,P11270120,null,null,null,null,0.012500,null,null,null
13508,FVN00156,null,P11270110,null,null,null,null,0.006188,null,null,null


### MOVE P,M TO THE LAST

In [102]:
def reposition_last_value(data_range, last_column_name):
    for idx in range(len(data_range)):
        row = data_range.iloc[idx]
        if row[last_column_name] != "null":
            continue  # Skip rows where the last column is not 'null'
        
        last_value = "null"
        last_column = "null"
        for col in reversed(data_range.columns):
            if row[col] != "null":
                last_value = row[col]
                last_column = col
                data_range.iat[idx, data_range.columns.get_loc(col)] = "null"
                break
        if last_value != "null" and last_column != "null":
            data_range.iat[idx, data_range.columns.get_loc(last_column_name)] = last_value
            data_range.iat[idx, data_range.columns.get_loc(last_column)] = "null"

# Assuming BOMtree_transformed_test is a DataFrame
BOMtree_transformed1 = BOMtree_transformed.iloc[:, :6]
BOMtree_transformed2 = BOMtree_transformed.iloc[:, 6:]
reposition_last_value(BOMtree_transformed1, 'Raw Material')
reposition_last_value(BOMtree_transformed2, 'ratio6')

In [103]:
# Assuming BOMtree_transformed_test1 and BOMtree_transformed_test2 are your DataFrames
BOMtree_transformed_layouted = pd.concat([BOMtree_transformed1, BOMtree_transformed2], axis=1)

# If you want to reset the index of the resulting DataFrame, you can do:
BOMtree_transformed_layouted = BOMtree_transformed_layouted.reset_index(drop=True)

# The resulting DataFrame (expanded_dataframe) will have columns from both input DataFrames side by side.
BOMtree_transformed_layouted

,SKU,3****,C toothbrush,C HDL,C HDL S.1,Raw Material,ratio2,ratio3,ratio4,ratio5,ratio6
0,VN01289A,null,C0000009305,null,null,M22411,null,18.0,null,null,0.000076
1,VN01289A,null,C0000009305,null,null,M44034,null,18.0,null,null,0.00051
2,VN01289A,null,C0000009305,null,null,M44761,null,18.0,null,null,0.000151
3,VN01289A,null,C0000009305,null,null,M44030,null,18.0,null,null,0.00025
4,VN01289A,null,C0000009305,C0000009257,null,M27839,null,18.0,1.0,null,0.002646
...,...,...,...,...,...,...,...,...,...,...,...
13505,FVN00156,null,null,null,null,P11241721,null,null,null,null,1.0
13506,FVN00156,null,null,null,null,P11270073,null,null,null,null,0.0009
13507,FVN00156,null,null,null,null,P11270120,null,null,null,null,0.0125
13508,FVN00156,null,null,null,null,P11270110,null,null,null,null,0.006188


### DEAL WITH HANDLE

In [104]:
df_list_handle = pd.read_csv(folder_path + "List Handle.csv", delimiter=',')
df_list_handle['SKU'] = df_list_handle['SKU'].astype(str)
df_list_handle = df_list_handle['SKU'].tolist()
df_list_handle

['1050458',
 '1050459',
 '1050460',
 '1050461',
 '61030124',
 '61030125',
 '61030126',
 '61030127',
 '61037835',
 '61037836']

In [105]:
for index, row in BOMtree_transformed_layouted.iterrows():
    if row['SKU'] in df_list_handle:
        BOMtree_transformed_layouted.at[index, 'C HDL S.1'] = row['C HDL']
        BOMtree_transformed_layouted.at[index, 'C HDL'] = row['C toothbrush']
        BOMtree_transformed_layouted.at[index, 'C toothbrush'] = 'null'
        BOMtree_transformed_layouted.at[index, 'ratio5'] = row['ratio4']
        BOMtree_transformed_layouted.at[index, 'ratio4'] = row['ratio3']
        BOMtree_transformed_layouted.at[index, 'ratio3'] = 'null'

In [106]:
BOMtree_transformed_layouted

,SKU,3****,C toothbrush,C HDL,C HDL S.1,Raw Material,ratio2,ratio3,ratio4,ratio5,ratio6
0,VN01289A,null,C0000009305,null,null,M22411,null,18.0,null,null,0.000076
1,VN01289A,null,C0000009305,null,null,M44034,null,18.0,null,null,0.00051
2,VN01289A,null,C0000009305,null,null,M44761,null,18.0,null,null,0.000151
3,VN01289A,null,C0000009305,null,null,M44030,null,18.0,null,null,0.00025
4,VN01289A,null,C0000009305,C0000009257,null,M27839,null,18.0,1.0,null,0.002646
...,...,...,...,...,...,...,...,...,...,...,...
13505,FVN00156,null,null,null,null,P11241721,null,null,null,null,1.0
13506,FVN00156,null,null,null,null,P11270073,null,null,null,null,0.0009
13507,FVN00156,null,null,null,null,P11270120,null,null,null,null,0.0125
13508,FVN00156,null,null,null,null,P11270110,null,null,null,null,0.006188


In [107]:
BOMtree_transformed_layouted.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13510 entries, 0 to 13509
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   SKU           13510 non-null  object
 1   3****         13510 non-null  object
 2   C toothbrush  13510 non-null  object
 3   C HDL         13510 non-null  object
 4   C HDL S.1     13510 non-null  object
 5   Raw Material  13510 non-null  object
 6   ratio2        13510 non-null  object
 7   ratio3        13510 non-null  object
 8   ratio4        13510 non-null  object
 9   ratio5        13510 non-null  object
 10  ratio6        13510 non-null  object
dtypes: object(11)
memory usage: 1.1+ MB


## 3.8 ADD INFORMATION OF P,M

In [108]:
BOMtree_temp = BOMtree
BOMtree_temp

,lv1,lv2,lv3,lv4,lv5,lv6
65,VN01289A,C0000009305,M22411,null,null,null
66,VN01289A,C0000009305,M44034,null,null,null
67,VN01289A,C0000009305,M44761,null,null,null
68,VN01289A,C0000009305,M44030,null,null,null
69,VN01289A,C0000009305,C0000009257,M27839,null,null
...,...,...,...,...,...,...
28668,FVN00156,P11241721,null,null,null,null
28669,FVN00156,P11270073,null,null,null,null
28670,FVN00156,P11270120,null,null,null,null
28671,FVN00156,P11270110,null,null,null,null


In [109]:
BOMtree_transformed_layouted = pd.merge(BOMtree_transformed_layouted, df[["Component",'Component Description','Component UOM']].drop_duplicates(subset=["Component"], keep="first"), how='left', left_on=['Raw Material'], right_on=['Component'])
# Drop 'Material' and 'Component' columns
BOMtree_transformed_layouted = BOMtree_transformed_layouted.drop(columns=['Component'])
BOMtree_transformed_layouted

,SKU,3****,C toothbrush,C HDL,C HDL S.1,Raw Material,ratio2,ratio3,ratio4,ratio5,ratio6,Component Description,Component UOM
0,VN01289A,null,C0000009305,null,null,M22411,null,18.0,null,null,0.000076,Hongfeng Aluminum 1.3X0.28MM anchor wire,KG
1,VN01289A,null,C0000009305,null,null,M44034,null,18.0,null,null,0.00051,K.R. PBT NONTAPER BLUE 305S 0.203MM/8MIL,KG
2,VN01289A,null,C0000009305,null,null,M44761,null,18.0,null,null,0.000151,K.R. PBT GREEN 328S 0.203MM/8MIL,KG
3,VN01289A,null,C0000009305,null,null,M44030,null,18.0,null,null,0.00025,K.R. PBT NONTAPER WHITE102S 0.203MM/8MIL,KG
4,VN01289A,null,C0000009305,C0000009257,null,M27839,null,18.0,1.0,null,0.002646,DONGGUAN TOP POLYMER 8201-601B0150004TPE,KG
...,...,...,...,...,...,...,...,...,...,...,...,...,...
13505,FVN00156,null,null,null,null,P11241721,null,null,null,null,1.0,EC 19 rPP FHM SHIPPER 1PK 48 LATAM,ST
13506,FVN00156,null,null,null,null,P11270073,null,null,null,null,0.0009,OPP TAPE WITH LOGO-910m,ROL
13507,FVN00156,null,null,null,null,P11270120,null,null,null,null,0.0125,Slip Sheet 40x48 0.8MM,ST
13508,FVN00156,null,null,null,null,P11270110,null,null,null,null,0.006188,STRETCH WRAP FILM WIDTH ROLL 500MM 14KG,KG


In [110]:
### new SKU

## 3.9 FILTER OUT LIST ONLY CHANGE QUANTITY

In [111]:
df_ecn_filtered

,Material,Component,Change no.,chng no. to,CmpValidFr,cmpValidTo,New/old Material
326,312272,C0000005899,NaN,500000028368,03/10/2023,04/17/2024,C0000010544 - C0000010545 - C0000010546 - C000...
327,312272,C0000010544,500000028368,NaN,04/17/2024,12/31/9999,C0000005899 - C0000005900 - C0000005901 - C000...
328,312272,C0000005900,NaN,500000028368,03/10/2023,04/17/2024,C0000010544 - C0000010545 - C0000010546 - C000...
329,312272,C0000010545,500000028368,NaN,04/17/2024,12/31/9999,C0000005899 - C0000005900 - C0000005901 - C000...
330,312272,C0000005901,NaN,500000028368,03/10/2023,04/17/2024,C0000010544 - C0000010545 - C0000010546 - C000...
...,...,...,...,...,...,...,...
29323,VN01129A,C0000010371,500000029326,NaN,11/26/2023,12/31/9999,C0000005930 - C0000005931 - C0000005932 - C000...
29327,VN01129A,P11270110,NaN,500000028821,02/14/2019,10/27/2023,P11270195
29328,VN01129A,P11270195,500000028821,NaN,10/27/2023,12/31/9999,P11270110
29460,VN01267A,P11270110,500000016876,500000028822,10/23/2018,10/27/2023,P11270195


In [112]:
df_change_quantity = df_ecn_filtered[df_ecn_filtered.duplicated(subset=['Material', 'Component'], keep=False)]
df_change_quantity.head()

,Material,Component,Change no.,chng no. to,CmpValidFr,cmpValidTo,New/old Material
12096,1053201,P11220265,500000014726,500000028314,05/08/2018,11/27/2023,P11212507 - P11220265 - P11270073
12097,1053201,P11220265,500000028314,NaN,11/27/2023,12/31/9999,P11211905 - P11220265 - P11270073
12100,1053201,P11270073,500000022195,500000028314,10/09/2020,11/27/2023,P11212507 - P11220265 - P11270073
12101,1053201,P11270073,500000028314,NaN,11/27/2023,12/31/9999,P11211905 - P11220265 - P11270073
16912,61016707,P11220274,500000023327,500000028925,07/12/2021,10/30/2023,P11212531 - P11220274


In [113]:
df_ecn_filtered = df_ecn_filtered.drop_duplicates(subset=['Material', 'Component'], keep=False)
len(df_ecn_filtered)

1098

## 3.10 ADD NEW MATERIAL

In [114]:
df_ecn_new_material = df_ecn_filtered[['Material','Component','New/old Material']]
df_ecn_new_material

,Material,Component,New/old Material
326,312272,C0000005899,C0000010544 - C0000010545 - C0000010546 - C000...
327,312272,C0000010544,C0000005899 - C0000005900 - C0000005901 - C000...
328,312272,C0000005900,C0000010544 - C0000010545 - C0000010546 - C000...
329,312272,C0000010545,C0000005899 - C0000005900 - C0000005901 - C000...
330,312272,C0000005901,C0000010544 - C0000010545 - C0000010546 - C000...
...,...,...,...
29323,VN01129A,C0000010371,C0000005930 - C0000005931 - C0000005932 - C000...
29327,VN01129A,P11270110,P11270195
29328,VN01129A,P11270195,P11270110
29460,VN01267A,P11270110,P11270195


In [115]:
BOMtree_temp = BOMtree

In [116]:
def left_merge_dataframes_new_material(left_col1, left_col2):
    global BOMtree_temp
    
    result_df = pd.merge(BOMtree_temp, df_ecn_new_material, how='left', left_on=[left_col1, left_col2], right_on=['Material', 'Component'])
    
    # Drop 'Material' and 'Component' columns
    result_df = result_df.drop(columns=['Material', 'Component'])
    
    # Rename columns with suffixes based on the input column names
    result_df = result_df.rename(columns={
        'New/old Material': 'New/old Material_' + left_col2
    })
    
    BOMtree_temp = result_df  # Update the global variable
    return BOMtree_temp

# Call the function for each pair of columns
left_merge_dataframes_new_material('lv1', 'lv2')
left_merge_dataframes_new_material('lv2', 'lv3')
left_merge_dataframes_new_material('lv3', 'lv4')
left_merge_dataframes_new_material('lv4', 'lv5')
left_merge_dataframes_new_material('lv5', 'lv6')

,lv1,lv2,lv3,lv4,lv5,lv6,New/old Material_lv2,New/old Material_lv3,New/old Material_lv4,New/old Material_lv5,New/old Material_lv6
0,VN01289A,C0000009305,M22411,null,null,null,NaN,NaN,NaN,NaN,NaN
1,VN01289A,C0000009305,M44034,null,null,null,NaN,NaN,NaN,NaN,NaN
2,VN01289A,C0000009305,M44761,null,null,null,NaN,NaN,NaN,NaN,NaN
3,VN01289A,C0000009305,M44030,null,null,null,NaN,NaN,NaN,NaN,NaN
4,VN01289A,C0000009305,C0000009257,M27839,null,null,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
13505,FVN00156,P11241721,null,null,null,null,NaN,NaN,NaN,NaN,NaN
13506,FVN00156,P11270073,null,null,null,null,NaN,NaN,NaN,NaN,NaN
13507,FVN00156,P11270120,null,null,null,null,NaN,NaN,NaN,NaN,NaN
13508,FVN00156,P11270110,null,null,null,null,P11270195,NaN,NaN,NaN,NaN


In [117]:
df_new_material_temp = BOMtree_temp[['New/old Material_lv2','New/old Material_lv3','New/old Material_lv4','New/old Material_lv5','New/old Material_lv6']]
df_new_material_temp

,New/old Material_lv2,New/old Material_lv3,New/old Material_lv4,New/old Material_lv5,New/old Material_lv6
0,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...
13505,NaN,NaN,NaN,NaN,NaN
13506,NaN,NaN,NaN,NaN,NaN
13507,NaN,NaN,NaN,NaN,NaN
13508,P11270195,NaN,NaN,NaN,NaN


In [118]:
# Move material that not start with 3 to the right 1 column
mask = df_new_material_temp['New/old Material_lv2'].apply(lambda x: isinstance(x, str) and not x.startswith('3'))
df_new_material_temp.loc[mask, 'New/old Material_lv6'] = df_new_material_temp.loc[mask, 'New/old Material_lv5']
df_new_material_temp.loc[mask, 'New/old Material_lv5'] = df_new_material_temp.loc[mask, 'New/old Material_lv4']
df_new_material_temp.loc[mask, 'New/old Material_lv4'] = df_new_material_temp.loc[mask, 'New/old Material_lv3']
df_new_material_temp.loc[mask, 'New/old Material_lv3'] = df_new_material_temp.loc[mask, 'New/old Material_lv2']
df_new_material_temp.loc[mask, 'New/old Material_lv2'] = np.nan
df_new_material_temp

C:\Users\Danh Bui\AppData\Local\Temp\ipykernel_11480\2987577924.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_new_material_temp.loc[mask, 'New/old Material_lv6'] = df_new_material_temp.loc[mask, 'New/old Material_lv5']
C:\Users\Danh Bui\AppData\Local\Temp\ipykernel_11480\2987577924.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_new_material_temp.loc[mask, 'New/old Material_lv5'] = df_new_material_temp.loc[mask, 'New/old Material_lv4']
C:\Users\Danh Bui\AppData\Local\Temp\ipykernel_11480\2987577924.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the 

,New/old Material_lv2,New/old Material_lv3,New/old Material_lv4,New/old Material_lv5,New/old Material_lv6
0,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...
13505,NaN,NaN,NaN,NaN,NaN
13506,NaN,NaN,NaN,NaN,NaN
13507,NaN,NaN,NaN,NaN,NaN
13508,NaN,P11270195,NaN,NaN,NaN


In [119]:
# Move starting with 'P' or 'M' to New/old Material_lv6
def assign_and_replace(column_name):
    # Create a copy of the DataFrame to avoid modifying the original data
    global df_new_material_temp

    # Assign values starting with 'P' or 'M' to New/old Material_lv6 and replace them with NaN in the specified column
    mask = df_new_material_temp[column_name].apply(lambda x: isinstance(x, str) and x.startswith(('P', 'M')))
    df_new_material_temp.loc[mask, 'New/old Material_lv6'] = df_new_material_temp.loc[mask, column_name].values
    df_new_material_temp.loc[mask, column_name] = np.nan

    return df_new_material_temp

In [120]:
# Move P, M to last column
assign_and_replace('New/old Material_lv3')
assign_and_replace('New/old Material_lv4')
assign_and_replace('New/old Material_lv5')

C:\Users\Danh Bui\AppData\Local\Temp\ipykernel_11480\2594630325.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_new_material_temp.loc[mask, 'New/old Material_lv6'] = df_new_material_temp.loc[mask, column_name].values
C:\Users\Danh Bui\AppData\Local\Temp\ipykernel_11480\2594630325.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_new_material_temp.loc[mask, column_name] = np.nan


,New/old Material_lv2,New/old Material_lv3,New/old Material_lv4,New/old Material_lv5,New/old Material_lv6
0,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...
13505,NaN,NaN,NaN,NaN,NaN
13506,NaN,NaN,NaN,NaN,NaN
13507,NaN,NaN,NaN,NaN,NaN
13508,NaN,NaN,NaN,NaN,P11270195


In [121]:
BOMtree_transformed_layouted = pd.concat([BOMtree_transformed_layouted, df_new_material_temp], axis=1)
BOMtree_transformed_layouted

,SKU,3****,C toothbrush,C HDL,C HDL S.1,Raw Material,ratio2,ratio3,ratio4,ratio5,ratio6,Component Description,Component UOM,New/old Material_lv2,New/old Material_lv3,New/old Material_lv4,New/old Material_lv5,New/old Material_lv6
0,VN01289A,null,C0000009305,null,null,M22411,null,18.0,null,null,0.000076,Hongfeng Aluminum 1.3X0.28MM anchor wire,KG,NaN,NaN,NaN,NaN,NaN
1,VN01289A,null,C0000009305,null,null,M44034,null,18.0,null,null,0.00051,K.R. PBT NONTAPER BLUE 305S 0.203MM/8MIL,KG,NaN,NaN,NaN,NaN,NaN
2,VN01289A,null,C0000009305,null,null,M44761,null,18.0,null,null,0.000151,K.R. PBT GREEN 328S 0.203MM/8MIL,KG,NaN,NaN,NaN,NaN,NaN
3,VN01289A,null,C0000009305,null,null,M44030,null,18.0,null,null,0.00025,K.R. PBT NONTAPER WHITE102S 0.203MM/8MIL,KG,NaN,NaN,NaN,NaN,NaN
4,VN01289A,null,C0000009305,C0000009257,null,M27839,null,18.0,1.0,null,0.002646,DONGGUAN TOP POLYMER 8201-601B0150004TPE,KG,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13505,FVN00156,null,null,null,null,P11241721,null,null,null,null,1.0,EC 19 rPP FHM SHIPPER 1PK 48 LATAM,ST,NaN,NaN,NaN,NaN,NaN
13506,FVN00156,null,null,null,null,P11270073,null,null,null,null,0.0009,OPP TAPE WITH LOGO-910m,ROL,NaN,NaN,NaN,NaN,NaN
13507,FVN00156,null,null,null,null,P11270120,null,null,null,null,0.0125,Slip Sheet 40x48 0.8MM,ST,NaN,NaN,NaN,NaN,NaN
13508,FVN00156,null,null,null,null,P11270110,null,null,null,null,0.006188,STRETCH WRAP FILM WIDTH ROLL 500MM 14KG,KG,NaN,NaN,NaN,NaN,P11270195


In [122]:
# Move to match position as relayout
for index, row in BOMtree_transformed_layouted.iterrows():
    if row['SKU'] in df_list_handle:
        BOMtree_transformed_layouted.at[index, 'New/old Material_lv5'] = row['New/old Material_lv4']
        BOMtree_transformed_layouted.at[index, 'New/old Material_lv4'] = row['New/old Material_lv3']
        BOMtree_transformed_layouted.at[index, 'New/old Material_lv3'] = row['New/old Material_lv2']

## 3.11 ADD CHANGE INFORMATION

In [123]:
BOMtree_temp = BOMtree

In [124]:
def left_join_with_concat_rename(input_column):
    global BOMtree_temp, df_ecn_filtered

    # Create a temporary DataFrame with the required columns from filtered_df_ecn1
    df_ecn_temp = df_ecn_filtered[['Material', 'Component', input_column]]

    # Perform left join and overwrite BOMtree_transformed_layouted
    for i in range(5):
        left_col = BOMtree_temp.columns[i]
        right_col = BOMtree_temp.columns[i + 1]

        # Calculate the new column name based on the current 'i'
        new_col_name = f'{input_column}{i + 2}'

        # Rename the input column in df_ecn_temp
        df_ecn_temp.columns = ['Material', 'Component', new_col_name]
        BOMtree_temp = pd.merge(BOMtree_temp, df_ecn_temp,
                                        left_on=[left_col, right_col],
                                        right_on=['Material', 'Component'],
                                        how='left')

        # Drop the 'Material' and 'Component' columns from BOMtree_transformed2
        BOMtree_temp.drop(['Material', 'Component'], axis=1, inplace=True)
        
    #BOMtree_temp[input_column] = BOMtree_temp.iloc[:, -5:].astype(str).replace('nan', '').apply(lambda row: ' - '.join(row), axis=1)
    BOMtree_temp[input_column] = BOMtree_temp.iloc[:, -5:].astype(str).apply(lambda row: ' - '.join(row[row != 'nan']), axis=1)
    
    last_column_index = BOMtree_temp.columns.get_loc(BOMtree_temp.columns[-1])
    columns_to_delete = BOMtree_temp.columns[last_column_index - 5:last_column_index]
    BOMtree_temp = BOMtree_temp.drop(columns=columns_to_delete)
    
    return BOMtree_temp


left_join_with_concat_rename('Change no.')
left_join_with_concat_rename('chng no. to')
left_join_with_concat_rename('CmpValidFr')
left_join_with_concat_rename('cmpValidTo')

,lv1,lv2,lv3,lv4,lv5,lv6,Change no.,chng no. to,CmpValidFr,cmpValidTo
0,VN01289A,C0000009305,M22411,null,null,null,,,,
1,VN01289A,C0000009305,M44034,null,null,null,,,,
2,VN01289A,C0000009305,M44761,null,null,null,,,,
3,VN01289A,C0000009305,M44030,null,null,null,,,,
4,VN01289A,C0000009305,C0000009257,M27839,null,null,,,,
...,...,...,...,...,...,...,...,...,...,...
13505,FVN00156,P11241721,null,null,null,null,,,,
13506,FVN00156,P11270073,null,null,null,null,,,,
13507,FVN00156,P11270120,null,null,null,null,,,,
13508,FVN00156,P11270110,null,null,null,null,,500000028804,03/10/2023,10/27/2023


In [125]:
df_change_info_temp = BOMtree_temp[['Change no.','chng no. to','CmpValidFr','cmpValidTo']]
df_change_info_temp

,Change no.,chng no. to,CmpValidFr,cmpValidTo
0,,,,
1,,,,
2,,,,
3,,,,
4,,,,
...,...,...,...,...
13505,,,,
13506,,,,
13507,,,,
13508,,500000028804,03/10/2023,10/27/2023


In [126]:
BOMtree_transformed_layouted = pd.concat([BOMtree_transformed_layouted, df_change_info_temp], axis=1)
BOMtree_transformed_layouted

,SKU,3****,C toothbrush,C HDL,C HDL S.1,Raw Material,ratio2,ratio3,ratio4,ratio5,...,Component UOM,New/old Material_lv2,New/old Material_lv3,New/old Material_lv4,New/old Material_lv5,New/old Material_lv6,Change no.,chng no. to,CmpValidFr,cmpValidTo
0,VN01289A,null,C0000009305,null,null,M22411,null,18.0,null,null,...,KG,NaN,NaN,NaN,NaN,NaN,,,,
1,VN01289A,null,C0000009305,null,null,M44034,null,18.0,null,null,...,KG,NaN,NaN,NaN,NaN,NaN,,,,
2,VN01289A,null,C0000009305,null,null,M44761,null,18.0,null,null,...,KG,NaN,NaN,NaN,NaN,NaN,,,,
3,VN01289A,null,C0000009305,null,null,M44030,null,18.0,null,null,...,KG,NaN,NaN,NaN,NaN,NaN,,,,
4,VN01289A,null,C0000009305,C0000009257,null,M27839,null,18.0,1.0,null,...,KG,NaN,NaN,NaN,NaN,NaN,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13505,FVN00156,null,null,null,null,P11241721,null,null,null,null,...,ST,NaN,NaN,NaN,NaN,NaN,,,,
13506,FVN00156,null,null,null,null,P11270073,null,null,null,null,...,ROL,NaN,NaN,NaN,NaN,NaN,,,,
13507,FVN00156,null,null,null,null,P11270120,null,null,null,null,...,ST,NaN,NaN,NaN,NaN,NaN,,,,
13508,FVN00156,null,null,null,null,P11270110,null,null,null,null,...,KG,NaN,NaN,NaN,NaN,P11270195,,500000028804,03/10/2023,10/27/2023


In [127]:
# Replace null with empty string
columns_to_replace = ['3****', 'C toothbrush', 'C HDL', 'C HDL S.1', 'Raw Material','ratio2','ratio3','ratio4','ratio5','ratio6']
BOMtree_transformed_layouted[columns_to_replace] = BOMtree_transformed_layouted[columns_to_replace].replace('null', '')
BOMtree_transformed_layouted

,SKU,3****,C toothbrush,C HDL,C HDL S.1,Raw Material,ratio2,ratio3,ratio4,ratio5,...,Component UOM,New/old Material_lv2,New/old Material_lv3,New/old Material_lv4,New/old Material_lv5,New/old Material_lv6,Change no.,chng no. to,CmpValidFr,cmpValidTo
0,VN01289A,,C0000009305,,,M22411,,18.0,,,...,KG,NaN,NaN,NaN,NaN,NaN,,,,
1,VN01289A,,C0000009305,,,M44034,,18.0,,,...,KG,NaN,NaN,NaN,NaN,NaN,,,,
2,VN01289A,,C0000009305,,,M44761,,18.0,,,...,KG,NaN,NaN,NaN,NaN,NaN,,,,
3,VN01289A,,C0000009305,,,M44030,,18.0,,,...,KG,NaN,NaN,NaN,NaN,NaN,,,,
4,VN01289A,,C0000009305,C0000009257,,M27839,,18.0,1.0,,...,KG,NaN,NaN,NaN,NaN,NaN,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13505,FVN00156,,,,,P11241721,,,,,...,ST,NaN,NaN,NaN,NaN,NaN,,,,
13506,FVN00156,,,,,P11270073,,,,,...,ROL,NaN,NaN,NaN,NaN,NaN,,,,
13507,FVN00156,,,,,P11270120,,,,,...,ST,NaN,NaN,NaN,NaN,NaN,,,,
13508,FVN00156,,,,,P11270110,,,,,...,KG,NaN,NaN,NaN,NaN,P11270195,,500000028804,03/10/2023,10/27/2023


In [128]:
BOMtree_transformed_layouted.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 13510 entries, 0 to 13509
Data columns (total 22 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   SKU                    13510 non-null  object 
 1   3****                  13510 non-null  object 
 2   C toothbrush           13510 non-null  object 
 3   C HDL                  13510 non-null  object 
 4   C HDL S.1              13510 non-null  object 
 5   Raw Material           13510 non-null  object 
 6   ratio2                 13510 non-null  object 
 7   ratio3                 13510 non-null  object 
 8   ratio4                 13510 non-null  object 
 9   ratio5                 13510 non-null  object 
 10  ratio6                 13510 non-null  float64
 11  Component Description  13510 non-null  object 
 12  Component UOM          13510 non-null  object 
 13  New/old Material_lv2   1347 non-null   object 
 14  New/old Material_lv3   3939 non-null   object 
 15  Ne

# 4 CONTINUE WITH OTHERS

## 4.1 SKU MASTER
https://docs.google.com/spreadsheets/d/1U1yERDQXrvE4_NbK_GwVIVvruC3oxWM90gLre0b6i9E/edit#gid=874534676

In [129]:
skumaster = pd.read_csv(folder_path + "SNP Master SKU YTD.csv", delimiter=',')
skumaster.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1290 entries, 0 to 1289
Data columns (total 3 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   VN SKU             1290 non-null   object
 1   Short Bundle Name  685 non-null    object
 2   Current Status     1290 non-null   object
dtypes: object(3)
memory usage: 30.4+ KB


In [130]:
#merge SKU master
BOMtree_transformed_layouted = BOMtree_transformed_layouted.merge(skumaster[['VN SKU', 'Short Bundle Name','Current Status']], how='left', left_on='SKU', right_on='VN SKU')
# BOMtree = BOMtree[(BOMtree["PLANT"] == 'VN11') & ~(BOMtree["PLANT_SP_MATL_STATUS"] == 'C5')]
# BOMtree = BOMtree[~(BOMtree["PLANT_SP_MATL_STATUS"] == 'C5')]
BOMtree_transformed_layouted = BOMtree_transformed_layouted.drop(columns=['VN SKU'])
BOMtree_transformed_layouted

,SKU,3****,C toothbrush,C HDL,C HDL S.1,Raw Material,ratio2,ratio3,ratio4,ratio5,...,New/old Material_lv3,New/old Material_lv4,New/old Material_lv5,New/old Material_lv6,Change no.,chng no. to,CmpValidFr,cmpValidTo,Short Bundle Name,Current Status
0,VN01289A,,C0000009305,,,M22411,,18.0,,,...,NaN,NaN,NaN,NaN,,,,,TA_21,Active
1,VN01289A,,C0000009305,,,M44034,,18.0,,,...,NaN,NaN,NaN,NaN,,,,,TA_21,Active
2,VN01289A,,C0000009305,,,M44761,,18.0,,,...,NaN,NaN,NaN,NaN,,,,,TA_21,Active
3,VN01289A,,C0000009305,,,M44030,,18.0,,,...,NaN,NaN,NaN,NaN,,,,,TA_21,Active
4,VN01289A,,C0000009305,C0000009257,,M27839,,18.0,1.0,,...,NaN,NaN,NaN,NaN,,,,,TA_21,Active
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13505,FVN00156,,,,,P11241721,,,,,...,NaN,NaN,NaN,NaN,,,,,EC_22RPP,Active
13506,FVN00156,,,,,P11270073,,,,,...,NaN,NaN,NaN,NaN,,,,,EC_22RPP,Active
13507,FVN00156,,,,,P11270120,,,,,...,NaN,NaN,NaN,NaN,,,,,EC_22RPP,Active
13508,FVN00156,,,,,P11270110,,,,,...,NaN,NaN,NaN,P11270195,,500000028804,03/10/2023,10/27/2023,EC_22RPP,Active


In [131]:
# Define the conditions for the new column
conditions = [
    BOMtree_transformed_layouted['chng no. to'].str.contains('|'.join(map(str, ECN_no))),
    BOMtree_transformed_layouted['Change no.'].str.contains('|'.join(map(str, ECN_no)))
]

# Define the choices for the new column
choices = ['Old', 'New']

# Create the new column based on the conditions and choices
BOMtree_transformed_layouted['old/new filter'] = pd.Series(pd.NA, index=BOMtree_transformed_layouted.index)  # Initialize the column with empty values
BOMtree_transformed_layouted['old/new filter'] = pd.Series(
    pd.Categorical(
        np.select(conditions, choices, default=''),
        categories=choices + [''],  # Include an empty category
        ordered=False
    ),
    index=BOMtree_transformed_layouted.index
)

In [132]:
kid_Decal = pd.read_csv(folder_path + "List Decal.csv", delimiter='\t')
kidDecal = kid_Decal.drop_duplicates()
kidDecal.head()

,Bundle Kid
0,K6+_OCEAN_21
1,K5+_TANDY
2,K5+_TAPER
3,K6+_JUNGLE_21


In [133]:
def get_rm_mother_list(row, kidDecal):
    if row['Short Bundle Name'] in kidDecal['Bundle Kid'].values:
        if row['C HDL S.1'] != '':
            return row['C HDL S.1']
        elif row['C HDL'] != '':
            return row['C HDL']
        elif row['C toothbrush'] != '':
            return row['C toothbrush']
        else:
            return row['SKU']
    else:
        if row['C HDL'] != '':
            return row['C HDL']
        elif row['C toothbrush'] != '':
            return row['C toothbrush']
        else:
            return row['SKU']

BOMtree_transformed_layouted['RM mother list'] = BOMtree_transformed_layouted.apply(lambda row: get_rm_mother_list(row, kidDecal), axis=1)

# Remove the column
column_to_move = BOMtree_transformed_layouted.pop('RM mother list')

# Insert the column at the desired position (index 6 for the 7th column)
BOMtree_transformed_layouted.insert(6, 'RM mother list', column_to_move)

In [134]:
# Define the lambda function to apply the conditions
def pm_ratio(row):
    if (row['3****'] != '') and (row['C toothbrush'] == '') and (row['C HDL'] == '') and (row['C HDL S.1'] == ''):
        return row['ratio2']
    else:
        return row['ratio6']

# Create the new column based on the conditions
BOMtree_transformed_layouted['PM ratio'] = BOMtree_transformed_layouted.apply(lambda row: pm_ratio(row), axis=1)

# Assuming your DataFrame is named BOMtree_transformed_layouted
column_to_move1 = BOMtree_transformed_layouted.pop('PM ratio')  # Remove the column

# Insert the column at the desired position (index 6 for the 7th column)
BOMtree_transformed_layouted.insert(12, 'PM ratio', column_to_move1)

## 4.2 NPI MASTER
https://docs.google.com/spreadsheets/d/1wlDX4puFnH314pqBmGSEBziVFJ3uBv_9IciuHF8r-90/edit#gid=1694214072

In [135]:
npi = pd.read_csv(folder_path + "NPI Master.csv", delimiter=',')
npi.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 352 entries, 0 to 351
Data columns (total 36 columns):
 #   Column                                            Non-Null Count  Dtype  
---  ------                                            --------------  -----  
 0   Date                                              352 non-null    object 
 1   Division request                                  280 non-null    object 
 2   Subs request                                      347 non-null    object 
 3   NPI Code link                                     352 non-null    object 
 4   NPI ID                                            352 non-null    int64  
 5   Project Status                                    352 non-null    object 
 6   Project name                                      352 non-null    object 
 7   SKU number of project                             352 non-null    object 
 8   New SKU                                           352 non-null    object 
 9   Refer SKU            

In [136]:
# Extract the date and time using regular expressions
npi['Estimate/Actual 1st shipment'] = npi['Estimate/Actual 1st shipment'].str[4:24]

# Convert the column to Timestamp
npi['Estimate/Actual 1st shipment'] = pd.to_datetime(npi['Estimate/Actual 1st shipment'], format='%b %d %Y %H:%M:%S')

In [137]:
# Convert today's date to a pandas Timestamp
today = pd.Timestamp(datetime.now().date())

# Calculate the date 50 days ago
days_ago = today - timedelta(days=30)

# Filter the DataFrame
filtered_npi = npi[(npi['Project Status'].isin(['Done', 'On-going'])) &
                   (npi['Estimate/Actual 1st shipment'] >= days_ago) &
                   ~(npi['New SKU'] == 'Not required') &
                   ~(npi['New SKU'] == '')]

filtered_npi.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 41 entries, 4 to 351
Data columns (total 36 columns):
 #   Column                                            Non-Null Count  Dtype         
---  ------                                            --------------  -----         
 0   Date                                              41 non-null     object        
 1   Division request                                  30 non-null     object        
 2   Subs request                                      41 non-null     object        
 3   NPI Code link                                     41 non-null     object        
 4   NPI ID                                            41 non-null     int64         
 5   Project Status                                    41 non-null     object        
 6   Project name                                      41 non-null     object        
 7   SKU number of project                             41 non-null     object        
 8   New SKU                        

In [138]:
#merge SKU master
BOMtree_transformed_layouted= BOMtree_transformed_layouted.merge(filtered_npi[['SKU number of project', 'New SKU']], how='left', left_on='SKU', right_on='SKU number of project')
BOMtree_transformed_layouted =BOMtree_transformed_layouted.drop(columns=['SKU number of project'])
BOMtree_transformed_layouted = BOMtree_transformed_layouted.rename(columns={'New SKU': 'New SKU NPI'})
BOMtree_transformed_layouted = BOMtree_transformed_layouted.merge(filtered_npi[['SKU number of project', 'New SKU']], how='left', left_on='SKU', right_on='New SKU')
BOMtree_transformed_layouted =BOMtree_transformed_layouted.drop(columns=['New SKU'])
BOMtree_transformed_layouted =BOMtree_transformed_layouted.rename(columns={'SKU number of project': 'Old SKU NPI'})
BOMtree_transformed_layouted

,SKU,3****,C toothbrush,C HDL,C HDL S.1,Raw Material,RM mother list,ratio2,ratio3,ratio4,...,New/old Material_lv6,Change no.,chng no. to,CmpValidFr,cmpValidTo,Short Bundle Name,Current Status,old/new filter,New SKU NPI,Old SKU NPI
0,VN01289A,,C0000009305,,,M22411,C0000009305,,18.0,,...,NaN,,,,,TA_21,Active,,NaN,NaN
1,VN01289A,,C0000009305,,,M44034,C0000009305,,18.0,,...,NaN,,,,,TA_21,Active,,NaN,NaN
2,VN01289A,,C0000009305,,,M44761,C0000009305,,18.0,,...,NaN,,,,,TA_21,Active,,NaN,NaN
3,VN01289A,,C0000009305,,,M44030,C0000009305,,18.0,,...,NaN,,,,,TA_21,Active,,NaN,NaN
4,VN01289A,,C0000009305,C0000009257,,M27839,C0000009257,,18.0,1.0,...,NaN,,,,,TA_21,Active,,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13505,FVN00156,,,,,P11241721,FVN00156,,,,...,NaN,,,,,EC_22RPP,Active,,NaN,NaN
13506,FVN00156,,,,,P11270073,FVN00156,,,,...,NaN,,,,,EC_22RPP,Active,,NaN,NaN
13507,FVN00156,,,,,P11270120,FVN00156,,,,...,NaN,,,,,EC_22RPP,Active,,NaN,NaN
13508,FVN00156,,,,,P11270110,FVN00156,,,,...,P11270195,,500000028804,03/10/2023,10/27/2023,EC_22RPP,Active,Old,NaN,NaN


## 4.3 LOST ALLOWANCE
df_marc[['MATERIAL', 'PLANT_SP_MATL_STATUS','PLANT','COMPONENT_SCRAP']]

In [139]:
BOMtree_transformed_layouted = BOMtree_transformed_layouted.merge(df_marc[['MATERIAL','COMPONENT_SCRAP']],how='left',left_on='Raw Material',right_on='MATERIAL')
BOMtree_transformed_layouted = BOMtree_transformed_layouted.drop(columns=['MATERIAL'])
BOMtree_transformed_layouted = BOMtree_transformed_layouted.rename(columns={'COMPONENT_SCRAP': 'LA (%)'})
BOMtree_transformed_layouted

,SKU,3****,C toothbrush,C HDL,C HDL S.1,Raw Material,RM mother list,ratio2,ratio3,ratio4,...,Change no.,chng no. to,CmpValidFr,cmpValidTo,Short Bundle Name,Current Status,old/new filter,New SKU NPI,Old SKU NPI,LA (%)
0,VN01289A,,C0000009305,,,M22411,C0000009305,,18.0,,...,,,,,TA_21,Active,,NaN,NaN,0.0
1,VN01289A,,C0000009305,,,M44034,C0000009305,,18.0,,...,,,,,TA_21,Active,,NaN,NaN,0.5
2,VN01289A,,C0000009305,,,M44761,C0000009305,,18.0,,...,,,,,TA_21,Active,,NaN,NaN,0.5
3,VN01289A,,C0000009305,,,M44030,C0000009305,,18.0,,...,,,,,TA_21,Active,,NaN,NaN,0.5
4,VN01289A,,C0000009305,C0000009257,,M27839,C0000009257,,18.0,1.0,...,,,,,TA_21,Active,,NaN,NaN,0.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13505,FVN00156,,,,,P11241721,FVN00156,,,,...,,,,,EC_22RPP,Active,,NaN,NaN,0.1
13506,FVN00156,,,,,P11270073,FVN00156,,,,...,,,,,EC_22RPP,Active,,NaN,NaN,2.0
13507,FVN00156,,,,,P11270120,FVN00156,,,,...,,,,,EC_22RPP,Active,,NaN,NaN,0.0
13508,FVN00156,,,,,P11270110,FVN00156,,,,...,,500000028804,03/10/2023,10/27/2023,EC_22RPP,Active,Old,NaN,NaN,3.0


## 4.4 PM BOM

In [140]:
P_RMfile = pd.read_csv(folder_path + "List P code RM.csv", delimiter=',')
P_RM = P_RMfile.drop_duplicates()
P_RM.head()

,P code RM
0,P11220259
1,P11220264
2,P11220265
3,P11220258
4,P11220288


In [141]:
# Filter BOMtree_transformed_layouted based on Raw Material starting with 'P' and not in P_RM
pmBOM = BOMtree_transformed_layouted[BOMtree_transformed_layouted['Raw Material'].str.startswith('P') & ~BOMtree_transformed_layouted['Raw Material'].isin(P_RM['P code RM']) & (BOMtree_transformed_layouted['old/new filter'] != "New")]

# Group by and aggregate to get the sum of PM ratio
pmBOM = pmBOM.groupby(['RM mother list', 'Raw Material', 'Component Description'], as_index=False)['PM ratio'].sum()

# Reset the index of pmBOM
pmBOM.reset_index(drop=True, inplace=True)

## 4.5 RM BOM

In [142]:
# Step 1: Filter rows based on the conditions
filtered_bom = BOMtree_transformed_layouted[
    (BOMtree_transformed_layouted['Raw Material'].str.startswith('P') == False) |
    (BOMtree_transformed_layouted['Raw Material'].isin(P_RM['P code RM']))
]

# Step 2: Select the desired columns
rmBOM = filtered_bom[['RM mother list', 'Raw Material', 'Component Description', 'ratio6']]

# Step 3: Drop duplicates based on the specified columns
rmBOM = rmBOM.drop_duplicates(subset=['RM mother list', 'Raw Material', 'Component Description', 'ratio6'])

# Now, rmBOM contains the desired data without affecting the original DataFrame.

In [144]:
rmBOM

,RM mother list,Raw Material,Component Description,ratio6
0,C0000009305,M22411,Hongfeng Aluminum 1.3X0.28MM anchor wire,0.000076
1,C0000009305,M44034,K.R. PBT NONTAPER BLUE 305S 0.203MM/8MIL,0.000510
2,C0000009305,M44761,K.R. PBT GREEN 328S 0.203MM/8MIL,0.000151
3,C0000009305,M44030,K.R. PBT NONTAPER WHITE102S 0.203MM/8MIL,0.000250
4,C0000009257,M27839,DONGGUAN TOP POLYMER 8201-601B0150004TPE,0.002646
...,...,...,...,...
13501,FVN00156,P11220274,PET ROLL 0.3x 273mm S6 HA,0.206320
13503,FVN00156,P11270079,SHRINK FILM-330mmx15mic-POF-0612,0.012600
13506,FVN00156,P11270073,OPP TAPE WITH LOGO-910m,0.000900
13507,FVN00156,P11270120,Slip Sheet 40x48 0.8MM,0.012500


## 4.6 AW TRACKING
https://docs.google.com/spreadsheets/d/1_eGB4Rz8ga7QXenUqIrzsSqbz-pKY6VWRuxMeuSdJIw/edit#gid=754673594

In [145]:
awtracking = pd.read_csv(folder_path + "AW Tracking.csv", delimiter=',')
awtracking.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 953 entries, 0 to 952
Data columns (total 46 columns):
 #   Column                                       Non-Null Count  Dtype  
---  ------                                       --------------  -----  
 0   Project                                      953 non-null    object 
 1   Code link                                    953 non-null    object 
 2   Division                                     868 non-null    object 
 3   BUNDLES                                      953 non-null    object 
 4   PACKING TYPE                                 953 non-null    object 
 5   VN.SKU                                       953 non-null    object 
 6   SUB.SKU                                      952 non-null    object 
 7   DESCRIPTION of SKU (SNP)                     949 non-null    object 
 8   Component                                    953 non-null    object 
 9   Current P                                    953 non-null    object 
 10  De

In [146]:
# Extract the date and time using regular expressions
awtracking['ECN DATE ESTIMATION'] = awtracking['ECN DATE ESTIMATION'].str[4:24]

# Convert the column to Timestamp
awtracking['ECN DATE ESTIMATION'] = pd.to_datetime(awtracking['ECN DATE ESTIMATION'], format='%b %d %Y %H:%M:%S')

ValueError: time data '-2021' does not match format '%b %d %Y %H:%M:%S' (match)

In [147]:
# Convert today's date to a pandas Timestamp
today1 = pd.Timestamp(datetime.now().date())

# Calculate the date 1 days ago
days_ago1 = today1 - timedelta(days=1)

# awtracking['ECN DATE ESTIMATION'] = pd.to_datetime(awtracking['ECN DATE ESTIMATION'], format="%m-%d-%Y", errors='coerce')

# Filter the DataFrame
awtracking_filter = awtracking[~(awtracking['Project'] == '') &
                    (awtracking['ECN DATE ESTIMATION'] >= days_ago1)]

awtracking_col = awtracking_filter[['VN.SKU', 'Current P', 'New SKU', 'New P', 'ECN# (PKG)', 'ECN DATE ESTIMATION']].copy()

# awtracking 1090
awtracking_col

TypeError: '>=' not supported between instances of 'str' and 'Timestamp'

In [148]:
# # Convert today's date to a pandas Timestamp
# today1 = pd.Timestamp(datetime.now().date())

# # Calculate the date 1 days ago
# days_ago1 = today1 - timedelta(days=1)

# awtracking['ECN DATE ESTIMATION'] = pd.to_datetime(awtracking['ECN DATE ESTIMATION'], format="%m-%d-%Y", errors='coerce')

# # Filter the DataFrame
# awtracking_filter = awtracking[~(awtracking['Project'] == '') &
#                     (awtracking['ECN DATE ESTIMATION'] >= days_ago1)]

# awtracking_col = awtracking_filter[['VN.SKU', 'Current P', 'New SKU', 'New P', 'ECN# (PKG)', 'ECN DATE ESTIMATION']].copy()

# # awtracking 1090
# awtracking_col

## 4.7 NON AW TRACKING
https://docs.google.com/spreadsheets/d/1_eGB4Rz8ga7QXenUqIrzsSqbz-pKY6VWRuxMeuSdJIw/edit#gid=1597922860

In [149]:
nonawtracking = pd.read_csv(folder_path + "Non AW Tracking.csv", delimiter=',')
# Extract the date and time using regular expressions
nonawtracking['ECN DATE ESTIMATION'] = nonawtracking['ECN DATE ESTIMATION'].str[4:24]

# Convert the column to Timestamp
nonawtracking['ECN DATE ESTIMATION'] = pd.to_datetime(nonawtracking['ECN DATE ESTIMATION'], format='%b %d %Y %H:%M:%S')
nonawtracking.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 857 entries, 0 to 856
Data columns (total 14 columns):
 #   Column                                       Non-Null Count  Dtype         
---  ------                                       --------------  -----         
 0   VN.SKU                                       857 non-null    object        
 1   SUB.SKU                                      857 non-null    object        
 2   DESCRIPTION of SKU (SNP)                     857 non-null    object        
 3   Component                                    834 non-null    object        
 4   Current P                                    782 non-null    object        
 5   Description of P# code                       786 non-null    object        
 6   New SKU                                      856 non-null    object        
 7   New P                                        779 non-null    object        
 8   ECN# (PKG)                                   829 non-null    object        
 9  

In [150]:
# nonawtracking['ECN DATE ESTIMATION'] = pd.to_datetime(nonawtracking['ECN DATE ESTIMATION'], format="%m-%d-%Y", errors='coerce')
# Filter the DataFrame
nonawtracking_filter = nonawtracking[~(nonawtracking['VN.SKU'] == '') &
                    (nonawtracking['ECN DATE ESTIMATION'] >= days_ago1)]

nonawtracking_col = nonawtracking_filter[['VN.SKU', 'Current P', 'New SKU', 'New P', 'ECN# (PKG)', 'ECN DATE ESTIMATION']].copy()

# awtracking 1090
nonawtracking_col

,VN.SKU,Current P,New SKU,New P,ECN# (PKG),ECN DATE ESTIMATION
188,1050545,C0000009478,Not required,C0000010385,500000027058,2023-10-31
189,1050545,C0000009479,Not required,C0000010386,500000027058,2023-10-31
190,1050545,C0000009480,Not required,C0000010387,500000027058,2023-10-31
191,1050545,C0000009481,Not required,C0000010388,500000027058,2023-10-31
192,1050587,C0000009478,Not required,C0000010385,500000027059,2023-10-31
...,...,...,...,...,...,...
779,61034569,C0000005900,Not required,C0000010545,500000028912,2023-12-17
780,61034569,C0000005901,Not required,C0000010546,500000028912,2023-12-17
781,61034569,C0000005902,Not required,C0000010547,500000028912,2023-12-17
782,1053201,P11220265,Not required,Not required,500000028314,2023-11-27


In [151]:
awtracking_concat = pd.concat([awtracking_col, nonawtracking_col])
awtracking_concat

NameError: name 'awtracking_col' is not defined

# 5 EXPORT OUTPUT

## 5.1 BOM

In [152]:
output_filename1 = 'MRP_BOM_master.xlsx'
output_path1 = os.path.join(folder_path, output_filename1)
BOMtree_transformed_layouted.to_excel(output_path1, index=False)

## 5.2 RECYCLE MATERIAL

In [153]:
output_filename2 = 'MRP_BOM_recycle.xlsx'
output_path2 = os.path.join(folder_path, output_filename2)
df_recycle.to_excel(output_path2, index=False)

## 5.3 BOM CHANGE QUANTITY

In [154]:
output_filename3 = 'MRP_BOM_consumptionchange.xlsx'
output_path3 = os.path.join(folder_path, output_filename3)
df_change_quantity.to_excel(output_path3, index=False)

## 5.4 AW TRACKING

In [446]:
output_filename4 = 'AWtracking.xlsx'
output_path4 = os.path.join(folder_path, output_filename4)
awtracking_concat.to_excel(output_path4, index=False)

## 5.5 RM BOM

In [232]:
output_filename5 = 'rmBOM.xlsx'
output_path5 = os.path.join(folder_path, output_filename5)
rmBOM.to_excel(output_path5, index=False)

## 5.6 PM BOM

In [233]:
output_filename6 = 'pmBOM.xlsx'
output_path6 = os.path.join(folder_path, output_filename6)
pmBOM.to_excel(output_path6, index=False)

In [234]:
print("DONE")

DONE
